# B2 — Clasificador Urbanístico

Notebook de clasificación taxonómica para publicaciones del dominio urbanístico.
Sigue la misma arquitectura incremental que B0 (ambiental-energético) y B1 (hídrico y natural).

In [1]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())

True

In [2]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIChatModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

In [3]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")

OK: qwen/qwen3.5-9b listo  |  otros: ['google/gemma-4-e4b', 'gemma-4-e4b-it', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']


## 1. Schema B2

In [4]:
from clasificador.schema_B2 import (
    ActType, CategoryTypeB2, SubcategoryTypeB2, ClassifierOutputB2
)

In [5]:
ejemplo_valido = ClassifierOutputB2(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    categories=[CategoryTypeB2.LIC_URB],
    subcategories=[SubcategoryTypeB2.USO_RUSTICO, SubcategoryTypeB2.USO_ENERGETICO],
    confidence=0.95,
    reasoning="'autorización de uso excepcional de suelo rústico' → LIC_URB + uso_rustico. 'planta solar fotovoltaica' → uso_energetico.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutputB2(
        is_relevant=False, act_type=ActType.RESOLUCION,
        categories=[CategoryTypeB2.PGOU], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "categories": [
    "LIC_URB"
  ],
  "subcategories": [
    "uso_rustico",
    "uso_energetico"
  ],
  "confidence": 0.95,
  "reasoning": "'autorización de uso excepcional de suelo rústico' → LIC_URB + uso_rustico. 'planta solar fotovoltaica' → uso_energetico."
}

Violación de invariante:
  ValidationError -> Value error, is_relevant=False con categories != []


## 2. Exploración del corpus B2

In [6]:
from clasificador.agent import get_ambito, inferir_act_type

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")

Corpus total: 65,201 registros | Columnas: ['id', 'pdf_link', 'expediente', 'promotor', 'proyecto', 'description', 'tipo', 'clean_id', 'bulletin', 'provincias', 'municipios', 'raw_scraped_timestamp', 'raw_scraped_year_month', 'scraped_timestamp', 'scraped_year_month', 'publication_timestamp', 'publication_type', 'contains_aau', 'proxy_pdf_link', 'ambito', 'rango']


In [7]:
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

keywords_dominio_B2 = [
    "plan general de ordenación", "pgou", "poum", "pom de ", "pxom", "normas subsidiarias",
    "normas urbanísticas municipales", "plan parcial", "plan especial", "estudio de detalle",
    "modificación puntual", "proyecto de urbanización", "reparcelación",
    "licencia urbanística", "autorización de uso excepcional", "calificación urbanística",
    "actuación específica de interés público", "declaración de interés comunitario",
    "declaración de utilidad e interés social",
]
mask_b2 = df["description"].str.lower().str.contains("|".join(keywords_dominio_B2), na=False)
df_b2 = df[mask_b2].copy()
print(f"Universo B2 estimado: {len(df_b2):,} registros ({len(df_b2)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución por boletín (top 10):")
print(df_b2["bulletin"].value_counts().head(10).to_string())
print(f"\nDistribución N1 en universo B2:")
print(df_b2["act_type_n1"].value_counts().head(8).to_string())

Universo B2 estimado: 1,687 registros (2.59% del corpus)

Distribución por boletín (top 10):
bulletin
bocyl    442
docm     202
bon      185
dogv     159
dogc     102
doe       78
dog       64
bopa      59
bocm      58
borm      56

Distribución N1 en universo B2:
act_type_n1
anuncio                623
información_pública    338
aprobación             259
resolución             211
acuerdo                 95
orden                   50
edicto                  41
corrección_errores      26


## 3. Ground truth — Muestreo estratificado

In [8]:
keywords_B2 = {
    "PGOU":     ["plan general de ordenación", "pgou", "poum", "pom de ", "pxom",
                 "pgom", "pgm de ", "normas subsidiarias",
                 "normas urbanísticas municipales", " num "],
    "PLAN_ESP": ["plan parcial", "plan especial", "estudio de detalle"],
    "MOD_PUN":  ["modificación puntual", "modificación del plan general",
                 "modificación de las normas subsidiarias",
                 "modificación de las normas urbanísticas"],
    "PROY_URB": ["proyecto de urbanización", "reparcelación", "parcelación",
                 "unidad de ejecución"],
    "LIC_URB":  ["licencia urbanística", "autorización de uso excepcional",
                 "calificación urbanística",
                 "actuación específica de interés público",
                 "usos y actividades admisibles en suelo rústico",
                 "autorización de actividades en suelo no urbanizable"],
    "DIC_INT":  ["declaración de interés comunitario",
                 "declaración de utilidad e interés social",
                 "uso de interés general"],
}

cuotas_B2 = {
    "LIC_URB": 30, "MOD_PUN": 25, "PLAN_ESP": 25,
    "PGOU": 25, "PROY_URB": 15, "DIC_INT": 5,
}
N_MULTILABEL_FV = 10
N_NEGATIVOS     = 15

desc_lower = df["description"].str.lower()
print(f"{'Label':<12} {'Pool':>8} {'Cuota':>7} {'Estado':>12}")
print("-" * 44)
for label, kws in keywords_B2.items():
    mask = desc_lower.str.contains("|".join(kws), na=False)
    pool = mask.sum()
    cuota = cuotas_B2.get(label, 0)
    estado = "OK" if pool >= cuota else f"REDUCIDA a {min(cuota, pool)}"
    print(f"{label:<12} {pool:>8,} {cuota:>7} {estado:>12}")

Label            Pool   Cuota       Estado
--------------------------------------------
PGOU              461      25           OK
PLAN_ESP          630      25           OK
MOD_PUN           445      25           OK
PROY_URB          231      15           OK
LIC_URB           391      30           OK
DIC_INT            30       5           OK


In [9]:
desc_lower = df["description"].str.lower()
sampled_ids = set()
frames = []

# Keywords para detectar renovables en suelo rústico (multilabel B2+B0)
kws_fv_rustico = ["fotovoltaica", "eólica", "aerogenerador", "solar"]
kws_rustico    = ["suelo rústico", "suelo no urbanizable", "uso excepcional"]

# 1. Muestra estratificada por categoría
for label, kws in keywords_B2.items():
    mask = desc_lower.str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas_B2.get(label, 0), len(pool))
    if n == 0:
        print(f"  {label:<12} SKIP (pool vacío)")
        continue
    sample = pool.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = label
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {label:<12} pool={len(pool):>5,}  sampled={n}")

# 2. Multilabel FV en suelo rústico (LIC_URB solar/eólica)
mask_fv = (
    desc_lower.str.contains("|".join(kws_fv_rustico), na=False)
    & desc_lower.str.contains("|".join(kws_rustico), na=False)
    & ~df.index.isin(sampled_ids)
)
pool_fv = df[mask_fv]
n_fv = min(N_MULTILABEL_FV, len(pool_fv))
if n_fv > 0:
    sample_fv = pool_fv.sample(n_fv, random_state=SEED).copy()
    sample_fv["grupo_muestreo"] = "MULTILABEL_FV"
    sampled_ids.update(sample_fv.index.tolist())
    frames.append(sample_fv)
    print(f"  {'MULTILABEL_FV':<12} pool={len(pool_fv):>5,}  sampled={n_fv}")

# 3. Negativos
all_kws_b2 = [kw for kws in keywords_B2.values() for kw in kws]
mask_neg = (
    ~desc_lower.str.contains("|".join(all_kws_b2), na=False)
    & ~df.index.isin(sampled_ids)
)
pool_neg = df[mask_neg]
negativos = pool_neg.sample(N_NEGATIVOS, random_state=SEED).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")
print(df_muestreo["grupo_muestreo"].value_counts().to_string())

  PGOU         pool=  461  sampled=25
  PLAN_ESP     pool=  629  sampled=25
  MOD_PUN      pool=  433  sampled=25
  PROY_URB     pool=  225  sampled=15
  LIC_URB      pool=  390  sampled=30
  DIC_INT      pool=   30  sampled=5
  MULTILABEL_FV pool=   74  sampled=10

Total muestreado: 150 registros
grupo_muestreo
LIC_URB          30
PGOU             25
PLAN_ESP         25
MOD_PUN          25
PROY_URB         15
NEGATIVO         15
MULTILABEL_FV    10
DIC_INT           5


In [10]:
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul  = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()

Validación del muestreo - 3 ejemplos por grupo:

--- PGOU ---
  [BOJA] Resolución de 16 de enero de 2025, de la Delegación Territorial de Fomento, Articulación del Territorio y Vivienda en Málaga, por 
  [BOCM] Plan general de urbanismo – Acuerdo de 12 de marzo de 2025, del Consejo de Gobierno, por el que se aprueba definitivamente la Deci
  [DOCM] Anuncio de 28/01/2025, del Ayuntamiento de Villar de Olalla (Cuenca), sobre exposición pública del Plan Especial de Reforma Interi

--- PLAN_ESP ---
  [BOPA] Resolución de 3 de febrero de 2025, de la Consejería de Transición Ecológica, Industria y Desarrollo Económico, por la que se form
  [BON] Aprobación definitiva del Plan Especial de Actuación Urbana de las unidades UE-9 y UE-10
  [BOCYL] INFORMACIÓN pública relativa a la aprobación inicial del estudio de detalle, promovido por «Decoración Muebles Cagigal, S.A.», par

--- MOD_PUN ---
  [BOCYL] ACUERDO de 19 de diciembre de 2024, del Pleno del Ayuntamiento de Palencia, por el que se aprue

In [11]:
PATH_MUESTREO = "../data/ground_truth/ground_truth_B2_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")

Guardado: ../data/ground_truth/ground_truth_B2_muestreo.csv  (150 registros)
Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt


## 4. Agente base

In [12]:
from clasificador.schema_B2 import ClassifierOutputB2
from clasificador.prompts_B2 import PROMPT_REGISTRY_B2
from clasificador.agent import build_agent, run_experiment

In [13]:
agent_b2_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

# Casos cualitativos representativos del dominio B2
casos_b2 = [
    ("Anuncio del Ayuntamiento de Torrelodones por el que se somete a información pública la aprobación inicial del Plan General de Ordenación Urbana.", "bocm"),
    ("Resolución de la Consejería de Urbanismo por la que se aprueba definitivamente la modificación puntual número 5 del Plan General Municipal de Cuenca.", "docm"),
    ("Anuncio de información pública relativo a la solicitud de autorización de uso excepcional de suelo rústico para instalación solar fotovoltaica de 2 MW en Villarrobledo (Albacete).", "docm"),
    ("Anuncio del Ajuntament de Girona pel qual se sotmet a informació pública l'aprovació inicial del Pla d'Ordenació Urbanística Municipal (POUM) de Girona.", "dogc"),
    ("Resolución de la Universidad Complutense de Madrid por la que se aprueba la modificación del plan de estudios del Grado en Medicina conforme al Real Decreto 822/2021.", "boe"),
]

for desc, bul in casos_b2:
    print(f"\n[{bul.upper()}] {desc[:80]}...")
    # result = await agent_b2_v1.run(f"Boletín: {bul.upper()}\n\nDescripción: {desc}")
    # print(result.output.model_dump_json(indent=2))


[BOCM] Anuncio del Ayuntamiento de Torrelodones por el que se somete a información públ...

[DOCM] Resolución de la Consejería de Urbanismo por la que se aprueba definitivamente l...

[DOCM] Anuncio de información pública relativo a la solicitud de autorización de uso ex...

[DOGC] Anuncio del Ajuntament de Girona pel qual se sotmet a informació pública l'aprov...

[BOE] Resolución de la Universidad Complutense de Madrid por la que se aprueba la modi...


## 5. Funciones de evaluación B2

In [14]:
def parse_labels(value) -> set:
    """Convierte cualquier representación de etiquetas a un set de strings."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    s = str(value).strip()
    if s.startswith("["):
        try:
            items = json.loads(s)
            return {str(i).strip('"') for i in items if i}
        except json.JSONDecodeError:
            pass
    return {v.strip().strip('"') for v in s.split(",") if v.strip()}

In [15]:
def compute_metrics_B2(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B2.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    ALL_CATS_B2 = [e.value for e in CategoryTypeB2]
    mlb = MultiLabelBinarizer(classes=ALL_CATS_B2)
    mlb.fit([ALL_CATS_B2])

    gt_labels   = [parse_labels(v) & set(ALL_CATS_B2) for v in df_eval["categories_gt"]]
    pred_labels = [parse_labels(v) & set(ALL_CATS_B2) for v in df_eval["categories_pred"]]
    Y    = mlb.transform(gt_labels)
    Yhat = mlb.transform(pred_labels)

    is_rel_gt   = df_eval["is_relevant_gt"].astype(bool)
    is_rel_pred = df_eval["is_relevant_pred"].astype(bool)

    exact = pd.Series([set(g) == set(p) for g, p in zip(gt_labels, pred_labels)])
    rel   = is_rel_gt

    metrics = {
        "is_rel_accuracy":  round(accuracy_score(is_rel_gt, is_rel_pred), 4),
        "is_rel_precision": round(precision_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_recall":    round(recall_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_f1":        round(f1_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "micro_f1":         round(f1_score(Y, Yhat, average="micro", zero_division=0), 4),
        "macro_f1":         round(f1_score(Y, Yhat, average="macro", zero_division=0), 4),
        "hamming_loss":     round(hamming_loss(Y, Yhat), 4),
        "jaccard_samples":  round(jaccard_score(Y, Yhat, average="samples", zero_division=0), 4),
        "subset_accuracy":  round(accuracy_score(Y, Yhat), 4),
        "exact": exact,
        "rel":   rel,
    }

    if verbose:
        title = f"-- {label} --" if label else "-- Métricas B2 --"
        print(f"\n{title}\n")
        tp = int((is_rel_gt & is_rel_pred).sum())
        fp = int((~is_rel_gt & is_rel_pred).sum())
        fn = int((is_rel_gt & ~is_rel_pred).sum())
        tn = int((~is_rel_gt & ~is_rel_pred).sum())
        print(f"is_relevant  Acc={metrics['is_rel_accuracy']:.3f}  P={metrics['is_rel_precision']:.3f}  "
              f"R={metrics['is_rel_recall']:.3f}  F1={metrics['is_rel_f1']:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {metrics['micro_f1']:.3f}")
        print(f"  Macro F1:      {metrics['macro_f1']:.3f}")
        print(f"  Hamming Loss:  {metrics['hamming_loss']:.4f}")
        print(f"  Jaccard:       {metrics['jaccard_samples']:.3f}")
        print(f"  Subset Acc:    {metrics['subset_accuracy']:.3f}\n")
        f1s = f1_score(Y, Yhat, average=None, zero_division=0)
        sups = Y.sum(axis=0)
        print(f"  {'Label':<12} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print(f"  {'-'*37}")
        for i, cat in enumerate(ALL_CATS_B2):
            prec = precision_score(Y[:, i], Yhat[:, i], zero_division=0)
            rec  = recall_score(Y[:, i], Yhat[:, i], zero_division=0)
            print(f"  {cat:<12} {prec:>6.3f} {rec:>6.3f} {f1s[i]:>6.3f} {int(sups[i]):>5}")
        print(f"  {'-'*37}")
        print(f"  {'Macro':<12} {'':>6} {'':>6} {metrics['macro_f1']:>6.3f}\n")

        cards = [len(g) for g in gt_labels]
        print(f"  Subset Acc por cardinalidad:")
        for card in [0, 1, 2]:
            idx = [i for i, c in enumerate(cards) if c == card]
            if idx:
                acc = accuracy_score(Y[idx], Yhat[idx])
                print(f"    card={card} ({'no relevante' if card == 0 else str(card)}) : {acc:.3f}  ({int(acc*len(idx))}/{len(idx)})")
        idx3 = [i for i, c in enumerate(cards) if c >= 3]
        if idx3:
            acc3 = accuracy_score(Y[idx3], Yhat[idx3])
            print(f"    card>=3               : {acc3:.3f}  ({int(acc3*len(idx3))}/{len(idx3)})")

        conf = df_eval.get("confidence", pd.Series(dtype=float))
        if conf.notna().any():
            print(f"\n  Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return metrics

In [16]:
def print_errors_B2(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    lbl = f" · {label}" if label else ""
    print(f"\n-- Errores N2 en relevantes{lbl} --")
    print(f"Total: {len(errores)}\n")
    for _, row in errores.iterrows():
        gt   = sorted(parse_labels(row["categories_gt"]))
        pred = sorted(parse_labels(row["categories_pred"]))
        falta = sorted(set(gt) - set(pred))
        sobra = sorted(set(pred) - set(gt))
        desc  = str(row["description"])[:90].replace("\n", " ")
        razon = str(row.get("reasoning", "")).replace("\n", " ")[:120]
        print(f"ID {row['id']} | GT={gt} | PRED={pred}")
        print(f"  Falta: {falta} | Sobra: {sobra}")
        print(f"  {desc}...")
        print(f"  Razonamiento: {razon}")
        print()

---

##  6. Experimento 1 - Baseline zero-shot

**Problema**: No existe un clasificador para el dominio urbanístico. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: Establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistematicos que guiarán la mejora del prompt.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V1 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [17]:
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B2_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"].notna()].copy()

df_b2_exp1 = await run_experiment(
    agent_b2_v1, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp1_baseline_qwen9b.csv",
    desc="B2 Exp1 - Baseline V1",
)
df_b2_exp1.head(3)

  Reanudando: 149/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boja,"Resolución de 16 de enero de 2025, de la Deleg...",PGOU,True,MOD_PUN,fase_definitiva,Publicacion registro Modificacion PGOU Mijas,True,orden,"[""PGOU"", ""MOD_PUN""]",[],0.95,La descripción menciona explícitamente «Modifi...,114.957
1,1,bocm,Plan general de urbanismo\n– Acuerdo de 12 de ...,PGOU,True,MOD_PUN,fase_definitiva,Decimoctava Modificacion Puntual PGOU Getafe,True,acuerdo,"[""PGOU"", ""MOD_PUN""]","[""fase_definitiva""]",0.98,"La descripción menciona explícitamente ""Plan g...",64.742
2,2,docm,"Anuncio de 28/01/2025, del Ayuntamiento de Vil...",PGOU,True,PLAN_ESP,fase_ip,PERIM exposicion publica - Plan Especial Refor...,True,anuncio,"[""PGOU"", ""PLAN_ESP""]","[""fase_ip""]",0.95,"El texto menciona ""Normas Subsidiarias"" que en...",71.333


In [18]:
df_b2_exp1 = pd.read_csv("../results/b2_exp1_baseline_qwen9b.csv")
df_eval_b2_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_1 = compute_metrics_B2(df_eval_b2_1, "Experimento B2-1 - Baseline")
print_errors_B2(df_eval_b2_1, m_b2_1["exact"], m_b2_1["rel"], label="B2 Experimento 1 - Baseline")


-- Experimento B2-1 - Baseline --

is_relevant  Acc=0.908  P=0.908  R=1.000  F1=0.952
             TP=138  FP=14  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.693
  Macro F1:      0.626
  Hamming Loss:  0.1206
  Jaccard:       0.532
  Subset Acc:    0.355

  Label             P      R     F1   Sup
  -------------------------------------
  PGOU          0.060  1.000  0.113     3
  PLAN_ESP      0.714  1.000  0.833    25
  MOD_PUN       0.977  0.977  0.977    44
  PROY_URB      0.833  0.652  0.732    23
  LIC_URB       0.767  0.825  0.795    40
  DIC_INT       0.179  1.000  0.303     5
  -------------------------------------
  Macro                       0.626

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.929  (13/14)
    card=1 (1) : 0.287  (39/136)
    card=2 (2) : 1.000  (2/2)

  Confianza: media=0.958  min=0.650  max=1.000

-- Errores N2 en relevantes · B2 Experimento 1 - Baseline --
Total: 97

ID 0 | GT=['MOD_PUN'] | PRED=['MOD_PUN', 'PGOU']
  Falta: [] | Sobra: ['

---

##  7. Experimento 2 - Prompt v2 (variantes regionales)

**Problema**: (rellenar tras el análisis de errores del §6)

**Objetivo**: Verificar si añadir la tabla de equivalencias regionales (POUM=PGOU, POM=PGOU, etc.) y la regla EMOT navarro mejora el F1 de PGOU.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V2 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [19]:
agent_b2_v2 = build_agent(
    model, "v2",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp2 = await run_experiment(
    agent_b2_v2, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp2_promptv2_qwen9b.csv",
    desc="B2 Exp2 - Prompt v2",
)
df_b2_exp2.head(3)

  Reanudando: 149/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boja,"Resolución de 16 de enero de 2025, de la Deleg...",PGOU,True,MOD_PUN,fase_definitiva,Publicacion registro Modificacion PGOU Mijas,True,resolución,"[""PGOU"", ""MOD_PUN""]",[],0.98,"La descripción menciona explícitamente ""Modifi...",179.477
1,1,bocm,Plan general de urbanismo\n– Acuerdo de 12 de ...,PGOU,True,MOD_PUN,fase_definitiva,Decimoctava Modificacion Puntual PGOU Getafe,True,acuerdo,"[""PGOU"", ""MOD_PUN""]","[""fase_definitiva""]",0.98,"El texto menciona explícitamente ""Plan General...",64.860
2,2,docm,"Anuncio de 28/01/2025, del Ayuntamiento de Vil...",PGOU,True,PLAN_ESP,fase_ip,PERIM exposicion publica - Plan Especial Refor...,True,anuncio,"[""PGOU"", ""PLAN_ESP""]","[""fase_ip"", ""uso_energetico"", ""uso_rustico""]",0.98,"El texto menciona ""Plan Especial de Reforma In...",137.210


In [20]:
df_b2_exp2 = pd.read_csv("../results/b2_exp2_promptv2_qwen9b.csv")
df_eval_b2_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_2 = compute_metrics_B2(df_eval_b2_2, "Experimento B2-2 - Prompt v2")
print_errors_B2(df_eval_b2_2, m_b2_2["exact"], m_b2_2["rel"], label="B2 Experimento 2 - Prompt v2")


-- Experimento B2-2 - Prompt v2 --

is_relevant  Acc=0.908  P=0.908  R=1.000  F1=0.952
             TP=138  FP=14  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.767
  Macro F1:      0.595
  Hamming Loss:  0.0735
  Jaccard:       0.630
  Subset Acc:    0.638

  Label             P      R     F1   Sup
  -------------------------------------
  PGOU          0.069  0.667  0.125     3
  PLAN_ESP      0.889  0.960  0.923    25
  MOD_PUN       1.000  0.841  0.914    44
  PROY_URB      0.895  0.739  0.810    23
  LIC_URB       0.857  0.750  0.800    40
  DIC_INT       0.000  0.000  0.000     5
  -------------------------------------
  Macro                       0.595

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 1.000  (14/14)
    card=1 (1) : 0.603  (82/136)
    card=2 (2) : 0.500  (1/2)

  Confianza: media=0.948  min=0.000  max=1.000

-- Errores N2 en relevantes · B2 Experimento 2 - Prompt v2 --
Total: 55

ID 0 | GT=['MOD_PUN'] | PRED=['MOD_PUN', 'PGOU']
  Falta: [] | Sobra: 

---

##  8. Experimento 3 - Prompt v3 (reglas de frontera)

**Problema**: (rellenar tras el análisis de errores del §7)

**Objetivo**: Verificar si las reglas de frontera (MOD_PUN≠PGOU, DIC_INT≠DUP, LIC_URB con renovables, PLAN_ESP≠PGOU) reducen los errores de clasificacion.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V3 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [21]:
agent_b2_v3 = build_agent(
    model, "v3",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp3 = await run_experiment(
    agent_b2_v3, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp3_promptv3_qwen9b.csv",
    desc="B2 Exp3 - Prompt v3",
)
df_b2_exp3.head(3)

  Reanudando: 149/150 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boja,"Resolución de 16 de enero de 2025, de la Deleg...",PGOU,True,MOD_PUN,fase_definitiva,Publicacion registro Modificacion PGOU Mijas,True,resolución,"[""MOD_PUN""]",[],0.98,"La publicación es una ""Modificación del Plan G...",121.850
1,1,bocm,Plan general de urbanismo\n– Acuerdo de 12 de ...,PGOU,True,MOD_PUN,fase_definitiva,Decimoctava Modificacion Puntual PGOU Getafe,True,acuerdo,"[""MOD_PUN""]","[""fase_definitiva""]",0.98,"El texto menciona explícitamente ""Decimoctava ...",52.278
2,2,docm,"Anuncio de 28/01/2025, del Ayuntamiento de Vil...",PGOU,True,PLAN_ESP,fase_ip,PERIM exposicion publica - Plan Especial Refor...,True,anuncio,"[""PGOU""]","[""fase_ip"", ""uso_equipamiento"", ""uso_rustico""]",0.85,"""exposición pública del Plan Especial de Refor...",59.080


In [22]:
df_b2_exp3 = pd.read_csv("../results/b2_exp3_promptv3_qwen9b.csv")
df_eval_b2_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_3 = compute_metrics_B2(df_eval_b2_3, "Experimento B2-3 - Prompt v3")
print_errors_B2(df_eval_b2_3, m_b2_3["exact"], m_b2_3["rel"], label="B2 Experimento 3 - Prompt v3")


-- Experimento B2-3 - Prompt v3 --

is_relevant  Acc=0.908  P=0.908  R=1.000  F1=0.952
             TP=138  FP=14  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.881
  Macro F1:      0.845
  Hamming Loss:  0.0351
  Jaccard:       0.760
  Subset Acc:    0.822

  Label             P      R     F1   Sup
  -------------------------------------
  PGOU          0.375  1.000  0.545     3
  PLAN_ESP      0.808  0.840  0.824    25
  MOD_PUN       1.000  0.955  0.977    44
  PROY_URB      1.000  0.783  0.878    23
  LIC_URB       0.968  0.750  0.845    40
  DIC_INT       1.000  1.000  1.000     5
  -------------------------------------
  Macro                       0.845

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 0.929  (13/14)
    card=1 (1) : 0.824  (112/136)
    card=2 (2) : 0.000  (0/2)

  Confianza: media=0.950  min=0.600  max=1.000

-- Errores N2 en relevantes · B2 Experimento 3 - Prompt v3 --
Total: 26

ID 2 | GT=['PLAN_ESP'] | PRED=['PGOU']
  Falta: ['PLAN_ESP'] | Sobra:

---

##  9. Experimento 4 - Prompt v4 (few-shot)

**Problema**: (rellenar tras el análisis de errores del §8)

**Objetivo**: Verificar si 5 ejemplos few-shot sobre los casos mas difíciles (MOD_PUN, PGOU variante regional, LIC_URB solar, falso positivo universidad, EMOT) eliminan los errores residuales.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V4 (V3 + 5 ejemplos) · few-shot · sin contexto N1.

**Resultados**: pendiente.

In [23]:
agent_b2_v4 = build_agent(
    model, "v4",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp4 = await run_experiment(
    agent_b2_v4, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp4_promptv4_qwen9b.csv",
    desc="B2 Exp4 - Prompt v4 few-shot",
)
df_b2_exp4.head(3)

  Reanudando: 149/150 registros ya clasificados

,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boja,"Resolución de 16 de enero de 2025, de la Deleg...",PGOU,True,MOD_PUN,fase_definitiva,Publicacion registro Modificacion PGOU Mijas,True,resolución,"[""MOD_PUN"", ""PGOU""]","[""fase_definitiva""]",0.95,"La publicación es una resolución sobre la ""Mod...",62.526
1,1,bocm,Plan general de urbanismo\n– Acuerdo de 12 de ...,PGOU,True,MOD_PUN,fase_definitiva,Decimoctava Modificacion Puntual PGOU Getafe,True,acuerdo,"[""MOD_PUN""]","[""fase_definitiva"", ""uso_residencial""]",0.95,"El texto menciona ""modificación puntual"" (frag...",73.765
2,2,docm,"Anuncio de 28/01/2025, del Ayuntamiento de Vil...",PGOU,True,PLAN_ESP,fase_ip,PERIM exposicion publica - Plan Especial Refor...,True,anuncio,"[""PGOU""]","[""fase_ip""]",0.95,"El texto menciona ""Plan Especial de Reforma In...",64.691


In [24]:
df_b2_exp4 = pd.read_csv("../results/b2_exp4_promptv4_qwen9b.csv")
df_eval_b2_4 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp4[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_4 = compute_metrics_B2(df_eval_b2_4, "Experimento B2-4 - Prompt v4 few-shot")
print_errors_B2(df_eval_b2_4, m_b2_4["exact"], m_b2_4["rel"], label="B2 Experimento 4 - Prompt v4 few-shot")


-- Experimento B2-4 - Prompt v4 few-shot --

is_relevant  Acc=0.934  P=1.000  R=0.927  F1=0.962
             TP=128  FP=0  FN=10  TN=14

N2 multilabel:
  Micro F1:      0.876
  Macro F1:      0.792
  Hamming Loss:  0.0373
  Jaccard:       0.763
  Subset Acc:    0.829

  Label             P      R     F1   Sup
  -------------------------------------
  PGOU          0.143  0.333  0.200     3
  PLAN_ESP      0.821  0.920  0.868    25
  MOD_PUN       0.976  0.909  0.941    44
  PROY_URB      1.000  0.739  0.850    23
  LIC_URB       0.944  0.850  0.895    40
  DIC_INT       1.000  1.000  1.000     5
  -------------------------------------
  Macro                       0.792

  Subset Acc por cardinalidad:
    card=0 (no relevante) : 1.000  (14/14)
    card=1 (1) : 0.824  (112/136)
    card=2 (2) : 0.000  (0/2)

  Confianza: media=0.950  min=0.750  max=1.000

-- Errores N2 en relevantes · B2 Experimento 4 - Prompt v4 few-shot --
Total: 26

ID 0 | GT=['MOD_PUN'] | PRED=['MOD_PUN', 'PGOU']
 

---

##  10. Comparativa de modelos - Gemma 4B con prompt V3

**Problema**: Todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos mas pequeños y rapidos.

**Objetivo**: Comprobar si Gemma 4 4B con V3 (no V4 por ventana de contexto) alcanza un rendimiento comparable al 9B.

**Enfoque**: Gemma 4 4B · SYSTEM_PROMPT_B2_V3 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio")
)
agent_b2_gemma = build_agent(
    model_gemma, "v3",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_gemma = await run_experiment(
    agent_b2_gemma, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp_gemma4b.csv",
    desc="B2 Gemma4B - Prompt v3",
)
df_b2_gemma.head(3)

  Reanudando: 2/150 registros ya clasificados


B2 Gemma4B - Prompt v3:   1%|          | 1/148 [14:26<35:22:44, 866.43s/it]


  [SKIP] context overflow: 'Anuncio de 28/01/2025, del Ayuntamiento de Villar de Olalla (Cuenca), sobre expo'


B2 Gemma4B - Prompt v3:   1%|▏         | 2/148 [14:44<14:53:40, 367.27s/it]


  [SKIP] context overflow: 'Anuncio de 27/12/2024, del Ayuntamiento de Albacete, de la Gerencia Municipal de'


In [ ]:
df_b2_gemma = pd.read_csv("../results/b2_exp_gemma4b.csv")
df_eval_b2_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_gemma = compute_metrics_B2(df_eval_b2_gemma, "B2 Gemma 4B - Prompt v3")
print_errors_B2(df_eval_b2_gemma, m_b2_gemma["exact"], m_b2_gemma["rel"], label="B2 Gemma 4B - Prompt v3")

KeyError: "['is_relevant_pred', 'act_type_pred', 'categories_pred', 'subcategories_pred', 'confidence', 'reasoning'] not in index"

## 11. Tabla resumen - Comparativa de experimentos B2

In [ ]:
experimentos_cfg_B2 = [
    ("B2 Exp1 - Baseline V1",    "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp1_baseline_qwen9b.csv"),
    ("B2 Exp2 - Prompt V2",      "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp2_promptv2_qwen9b.csv"),
    ("B2 Exp3 - Prompt V3",      "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp3_promptv3_qwen9b.csv"),
    ("B2 Exp4 - Prompt V4",      "Qwen 3.5 9B", "Few-shot",  "../results/b2_exp4_promptv4_qwen9b.csv"),
    ("B2 Exp5 - Gemma 4B V3",    "Gemma 4 4B",  "Zero-shot", "../results/b2_exp_gemma4b.csv"),
]

rows_b2 = []
metrics_list_b2 = []
for nombre, modelo, config, path in experimentos_cfg_B2:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B2(df_e, verbose=False)
    metrics_list_b2.append({"Experimento": nombre, **m})
    rows_b2.append({
        "Experimento":     nombre,
        "Modelo":          modelo,
        "Config":          config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b2 = pd.DataFrame(rows_b2)
df_display_b2 = df_summary_b2.copy()
df_display_b2.columns = [
    "Experimento", "Modelo", "Config",
    "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
    "s/item", "Total (s)",
]
display(df_display_b2.set_index("Experimento"))